# Change-data-capture on a trigger topic — publish, filtered subscribe, resume

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/13-cdc/cdc.ipynb)

Built from [`cookbook/book/chapters/13-cdc/cdc.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/13-cdc/cdc.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. On that GPU the
# chapter runs at `full` scale, over the published data; set SCALE = "small" to
# run the seconds-long version over the committed fixtures instead.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "full" if gpu else "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `register_topic` · `publish_topic` · `subscribe_collect`
(`predicate` / `from_offset` / `replay_only` / `max_batches`) · `list_topics` ·
**Theory:** the offset-addressed, append-only commit log — Kafka-style publish →
replay-from-offset (Kreps et al. 2011) — as the streaming complement of the
append-only result log (Kleppmann 2017) · **Rail:** measurement (every
collected stream equals the change set it claims to carry, counted from the
source tables).

The result log writes derived results once, append-only (Kleppmann 2017). A
**trigger topic** is its streaming complement: an offset-addressed, append-only
**commit log** of *events*. You `register_topic` with a row schema and
`publish_topic` batches onto it, each landing at the next 0-based offset; a
consumer collects by **replaying** the log from an offset it chose. That is the
Kafka model (Kreps et al. 2011), and it is the substrate change-data-capture (CDC)
rides: a stream of record changes that a downstream subscribes to selectively and
resumes from a checkpoint.

The changes here are real: the ogbn-arxiv corpus as it grew. Each publication
year is one batch of events — an `add` for every paper published that year and a
`cite` for every citation those papers make. The embedded `Database` runs the
topic on its in-process broker, with no server and no external broker.

In [ ]:
import tempfile
import threading

import jammi
import pyarrow as pa
import pyarrow.compute as pc
from jammi_cookbook import contracts, datasets, scale

SCALE = scale.current()
db = jammi.connect(f"file://{tempfile.mkdtemp()}")
arxiv = datasets.arxiv(db, SCALE)
PAPERS = f"{arxiv.papers}.public.{arxiv.papers}"
CITES = f"{arxiv.cites}.public.{arxiv.cites}"

## Register the topic and publish the corpus's history

A change event names its operation, the paper it concerns, and the year it
happened; `ref` is the paper's subject for an `add` and the cited paper for a
`cite`.

In [ ]:
SCHEMA = pa.schema([
    ("op", pa.string()),
    ("paper_id", pa.string()),
    ("year", pa.int64()),
    ("ref", pa.string()),
])
db.register_topic("paper_changes", schema=SCHEMA)
print(db.list_topics())

changes = db.sql(
    f"SELECT 'add' AS op, paper_id, year, subject AS ref FROM {PAPERS} "
    f"UNION ALL SELECT 'cite' AS op, c.src AS paper_id, p.year, c.dst AS ref "
    f"FROM {CITES} c JOIN {PAPERS} p ON c.src = p.paper_id"
).cast(SCHEMA)
years = sorted(set(changes.column("year").to_pylist()))
offset_of = {
    year: db.publish_topic("paper_changes", batch=changes.filter(pc.equal(changes["year"], year)))
    for year in years
}
print(f"{changes.num_rows} events in {len(years)} yearly batches, offsets 0..{len(years) - 1}")

In [ ]:
assert list(offset_of.values()) == list(range(len(years)))  # each publish takes the next offset

## Replay the log — a collect is a finite drain

`subscribe_collect` replays the topic's backing table from `from_offset` and
returns: every batch published so far, as one table. The log is the source's
history, so the replay holds exactly one event per paper and one per citation.

In [ ]:
def rows(sql: str) -> int:
    return db.sql(sql).column(0)[0].as_py()


papers, cites = rows(f"SELECT COUNT(*) FROM {PAPERS}"), rows(f"SELECT COUNT(*) FROM {CITES}")
replay = db.subscribe_collect("paper_changes", from_offset=0)
print(f"replayed {replay.num_rows} events = {papers} papers + {cites} citations")

In [ ]:
assert replay.num_rows == papers + cites
contracts.assert_close("cdc.events", replay.num_rows)

## A selective subscribe — only the new papers

A downstream that indexes papers wants the `add`s alone. The `predicate` is SQL
over the topic's schema, applied as the log is read; a batch with no matching
row is not delivered at all.

In [ ]:
adds = db.subscribe_collect("paper_changes", predicate="op = 'add'", from_offset=0)
print(f"{adds.num_rows} add events; the first few:")
for event in adds.slice(0, 3).to_pylist():
    print(" ", event)

In [ ]:
assert adds.num_rows == papers
assert set(adds.column("op").to_pylist()) == {"add"}

## Resume from a checkpoint

A CDC consumer stores its progress as an offset and resumes from it. One that
has processed the corpus through 2018 resumes at the first test-era batch, and
reads exactly the changes it has not seen.

In [ ]:
CHECKPOINT = offset_of[min(y for y in years if y >= 2019)]
tail = db.subscribe_collect("paper_changes", from_offset=CHECKPOINT)
unseen = rows(
    f"SELECT (SELECT COUNT(*) FROM {PAPERS} WHERE year >= 2019) + "
    f"(SELECT COUNT(*) FROM {CITES} c JOIN {PAPERS} p ON c.src = p.paper_id WHERE p.year >= 2019)"
)
print(f"resumed at offset {CHECKPOINT}: {tail.num_rows} events ({unseen} changes since 2018)")

In [ ]:
assert tail.num_rows == unseen
contracts.assert_close("cdc.tail_events", tail.num_rows)

## Resume *and* filter — a downstream feature, kept current

The two compose. A consumer maintaining each paper's citation in-degree (the
feature of the feature-store chapter) resumes at its checkpoint and reads only
the `cite` events: each is one unit of in-degree for the paper it names in
`ref`. Folded, the stream gives exactly the in-degree the test era added.

In [ ]:
new_cites = db.subscribe_collect(
    "paper_changes", predicate="op = 'cite'", from_offset=CHECKPOINT
)
increments = new_cites.group_by("ref").aggregate([("ref", "count")])
from_source = db.sql(
    f"SELECT c.dst AS ref, COUNT(*) AS n FROM {CITES} c "
    f"JOIN {PAPERS} p ON c.src = p.paper_id WHERE p.year >= 2019 GROUP BY c.dst"
).to_pylist()
agree = dict(zip(increments["ref"].to_pylist(), increments["ref_count"].to_pylist())) == {
    r["ref"]: r["n"] for r in from_source
}
print(f"{new_cites.num_rows} new citations across {increments.num_rows} papers; "
      f"stream agrees with the source: {agree}")

In [ ]:
assert agree
contracts.assert_close("cdc.tail_cites", new_cites.num_rows)

## Following the live tail

A replay ends; a live consumer does not. With `replay_only=False` the collect
reads the replay and then **follows** the broker, waiting for what is published
next, and returns once `max_batches` batches have arrived. The tail never ends
on its own, so `max_batches` is required; without it the call is refused rather
than left to wait forever. Here a follower waits on the next offset while a
correction — a paper retracted — is published.

In [ ]:
next_offset = len(years)
followed = {}
follower = threading.Thread(
    target=lambda: followed.setdefault(
        "batch",
        db.subscribe_collect(
            "paper_changes", from_offset=next_offset, replay_only=False, max_batches=1
        ),
    )
)
follower.start()
retracted = adds.column("paper_id")[0].as_py()
db.publish_topic(
    "paper_changes",
    batch=pa.table({"op": ["retract"], "paper_id": [retracted], "year": [2020], "ref": [None]},
                   schema=SCHEMA),
)
follower.join(timeout=60)
print(followed["batch"].to_pylist())

In [ ]:
assert followed["batch"].column("op").to_pylist() == ["retract"]

In [ ]:
db.close()

## Bridge note

> **A trigger topic is an offset-addressed commit log.** The engine writes
> derived results as an append-only log of immutable Parquet
> (Kleppmann 2017); a topic is the streaming complement — an append-only log
> of *events* a consumer replays from an offset it stored, the Kafka model
> (Kreps et al. 2011). CDC rides it directly: publish a source's changes, collect
> them with a predicate, resume from a checkpoint, or follow the live tail. The
> measured property is the one CDC depends on — the collected stream is exactly
> the change set it claims to carry: every paper, every citation, the unseen
> tail, and the feature it folds into.

## References

- Kreps, Jay, Narkhede, Neha, Rao, Jun (2011) *Kafka: A Distributed Messaging System for Log Processing* Proceedings of the NetDB Workshop on Networking Meets Databases.
- Kleppmann, Martin (2017) *Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems* O'Reilly Media.